In [2]:

import json
import pandas as pd
from pathlib import Path

D2_ROOT = Path(
    "/kaggle/input/datasets/nayebul0/"
    "assignment02-d2-rice-leaf-5932/"
    "Rice Leaf Disease Images"
)

SPLIT_ROOT = Path(
    "/kaggle/input/datasets/nayebul0/"
    "assignment02-verified-train-val-test-splits"
)

d2_manifest_path = SPLIT_ROOT / "D2_split_manifest.csv"
class_map_path = SPLIT_ROOT / "class_maps.json"

assert D2_ROOT.is_dir()
assert d2_manifest_path.is_file()
assert class_map_path.is_file()

d2_manifest = pd.read_csv(d2_manifest_path)

class_maps = json.loads(
    class_map_path.read_text(encoding="utf-8")
)

d2_class_map = class_maps["D2"]

expected_splits = {
    "train": 3327,
    "val": 712,
    "test": 713
}

actual_splits = (
    d2_manifest["split"].value_counts().to_dict()
)

assert len(d2_manifest) == 4752
assert actual_splits == expected_splits
assert d2_manifest["rel_path"].is_unique
assert d2_manifest["rel_path"].notna().all()
assert set(d2_manifest["class_id"]) == set(range(4))

assert d2_class_map == {
    "Bacterial Blight": 0,
    "Blast": 1,
    "Brown Spot": 2,
    "Tungro": 3
}

missing = [
    str(D2_ROOT / str(path))
    for path in d2_manifest["rel_path"]
    if not (D2_ROOT / str(path)).is_file()
]

assert not missing, f"Missing files: {missing[:5]}"

print("Dataset: D2")
print("Retained images:", len(d2_manifest))
print("Train:", actual_splits["train"])
print("Validation:", actual_splits["val"])
print("Test:", actual_splits["test"])
print("Missing images:", len(missing))

print("\nD2 CLASS MAPPING")
for name, class_id in sorted(
    d2_class_map.items(),
    key=lambda item: item[1]
):
    print(class_id, name)

print("\nPAPER 6 D2 DATASET VERIFICATION: PASSED")


Dataset: D2
Retained images: 4752
Train: 3327
Validation: 712
Test: 713
Missing images: 0

D2 CLASS MAPPING
0 Bacterial Blight
1 Blast
2 Brown Spot
3 Tungro

PAPER 6 D2 DATASET VERIFICATION: PASSED


In [3]:

import torch
import torch.nn as nn

from torchvision.models import (
    squeezenet1_1,
    SqueezeNet1_1_Weights
)

torch.manual_seed(2026)


class Paper6MFASqueezeNet(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()

        backbone = squeezenet1_1(
            weights=SqueezeNet1_1_Weights.IMAGENET1K_V1
        )

        self.features = backbone.features

        for index in (0, 3, 4):
            for parameter in self.features[index].parameters():
                parameter.requires_grad = False

        self.dropout = backbone.classifier[0]

        self.mfa_1x1 = nn.Conv2d(
            in_channels=512,
            out_channels=5,
            kernel_size=1
        )

        self.mfa_3x3 = nn.Conv2d(
            in_channels=512,
            out_channels=5,
            kernel_size=3,
            padding=1
        )

        self.relu = nn.ReLU(inplace=False)

        self.class_projection = nn.Conv2d(
            in_channels=10,
            out_channels=num_classes,
            kernel_size=1
        )

        self.pool = nn.AdaptiveAvgPool2d((1, 1))

    def forward(self, x):
        features = self.features(x)
        features = self.dropout(features)

        path1 = self.relu(self.mfa_1x1(features))
        path2 = self.relu(self.mfa_3x3(features))

        combined = torch.cat(
            (path1, path2),
            dim=1
        )

        class_maps = self.class_projection(combined)
        pooled = self.pool(class_maps)

        return torch.flatten(pooled, 1)


model = Paper6MFASqueezeNet(num_classes=4)

model.eval()

x = torch.randn(1, 3, 227, 227)

with torch.inference_mode():
    features = model.features(x)

    path1 = model.mfa_1x1(features)
    path2 = model.mfa_3x3(features)

    combined = torch.cat(
        (path1, path2),
        dim=1
    )

    class_maps = model.class_projection(combined)
    logits = model(x)

assert features.shape == (1, 512, 14, 14)
assert path1.shape == (1, 5, 14, 14)
assert path2.shape == (1, 5, 14, 14)
assert combined.shape == (1, 10, 14, 14)
assert class_maps.shape == (1, 4, 14, 14)
assert logits.shape == (1, 4)

total_parameters = sum(
    p.numel() for p in model.parameters()
)

frozen_parameters = sum(
    p.numel()
    for p in model.parameters()
    if not p.requires_grad
)

trainable_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

assert total_parameters == 748150
assert frozen_parameters == 25632
assert trainable_parameters == 722518

model.train()
model.zero_grad(set_to_none=True)

training_logits = model(x)
target = torch.tensor([3])

loss = nn.CrossEntropyLoss()(
    training_logits,
    target
)

loss.backward()

assert torch.isfinite(loss)

assert model.features[0].weight.grad is None
assert model.features[3].squeeze.weight.grad is None
assert model.features[4].squeeze.weight.grad is None

assert model.features[6].squeeze.weight.grad is not None
assert model.mfa_1x1.weight.grad is not None
assert model.mfa_3x3.weight.grad is not None
assert model.class_projection.weight.grad is not None

assert all(
    p.grad is None or torch.isfinite(p.grad).all()
    for p in model.parameters()
)

print("Backbone features:", tuple(features.shape))
print("MFA 1x1 branch:", tuple(path1.shape))
print("MFA 3x3 branch:", tuple(path2.shape))
print("Concatenated features:", tuple(combined.shape))
print("Classification logits:", tuple(logits.shape))

print("\nTotal parameters:", f"{total_parameters:,}")
print("Frozen parameters:", f"{frozen_parameters:,}")
print("Trainable parameters:", f"{trainable_parameters:,}")

print("\nForward pass: PASSED")
print("Gradient verification: PASSED")
print("Layer freezing: PASSED")

print("\nPAPER 6 D2 ARCHITECTURE: PASSED")


Downloading: "https://download.pytorch.org/models/squeezenet1_1-b8a52dc0.pth" to /root/.cache/torch/hub/checkpoints/squeezenet1_1-b8a52dc0.pth


100%|██████████| 4.73M/4.73M [00:00<00:00, 67.2MB/s]


Backbone features: (1, 512, 14, 14)
MFA 1x1 branch: (1, 5, 14, 14)
MFA 3x3 branch: (1, 5, 14, 14)
Concatenated features: (1, 10, 14, 14)
Classification logits: (1, 4)

Total parameters: 748,150
Frozen parameters: 25,632
Trainable parameters: 722,518

Forward pass: PASSED
Gradient verification: PASSED
Layer freezing: PASSED

PAPER 6 D2 ARCHITECTURE: PASSED


In [4]:

import random
import cv2
import numpy as np
import torch

from PIL import Image
from pathlib import Path
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, RandomSampler, SequentialSampler


def paper6_lab_minmax(image):
    rgb = np.asarray(
        image.convert("RGB"),
        dtype=np.float32
    ) / 255.0

    lab = cv2.cvtColor(
        rgb,
        cv2.COLOR_RGB2LAB
    )

    lightness = lab[:, :, 0].copy()

    minimum = float(lightness.min())
    maximum = float(lightness.max())

    if maximum > minimum:
        lab[:, :, 0] = (
            100.0
            * (lightness - minimum)
            / (maximum - minimum)
        )

    enhanced_rgb = cv2.cvtColor(
        lab,
        cv2.COLOR_LAB2RGB
    )

    enhanced_rgb = np.rint(
        np.clip(enhanced_rgb, 0.0, 1.0) * 255.0
    ).astype(np.uint8)

    return Image.fromarray(enhanced_rgb)


def paper6_augment(image, rng):
    array = np.array(image.convert("RGB"))
    height, width = array.shape[:2]

    flipped = rng.random() < 0.5

    if flipped:
        array = cv2.flip(array, 1)

    scale_x = rng.uniform(0.9, 1.1)
    scale_y = rng.uniform(0.9, 1.1)

    translate_x = rng.uniform(-30, 30)
    translate_y = rng.uniform(-30, 30)

    center_x = (width - 1) / 2
    center_y = (height - 1) / 2

    matrix = np.array([
        [
            scale_x,
            0.0,
            (1.0 - scale_x) * center_x + translate_x
        ],
        [
            0.0,
            scale_y,
            (1.0 - scale_y) * center_y + translate_y
        ]
    ], dtype=np.float32)

    augmented = cv2.warpAffine(
        array,
        matrix,
        (width, height),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=(0, 0, 0)
    )

    parameters = {
        "horizontal_flip": flipped,
        "scale_x": scale_x,
        "scale_y": scale_y,
        "translate_x_px": translate_x,
        "translate_y_px": translate_y
    }

    return Image.fromarray(augmented), parameters


class Paper6Transform:
    def __init__(self, training, seed=2026):
        self.training = training
        self.rng = random.Random(seed)
        self.to_tensor = transforms.ToTensor()
        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )

    def __call__(self, image):
        image = image.convert("RGB").resize(
            (227, 227),
            Image.Resampling.BILINEAR
        )

        image = paper6_lab_minmax(image)

        if self.training:
            image, _ = paper6_augment(
                image,
                self.rng
            )

        return self.normalize(
            self.to_tensor(image)
        )


class Paper6RiceDataset(Dataset):
    def __init__(self, rows, image_root, transform):
        self.rows = rows.reset_index(drop=True).copy()
        self.root = Path(image_root)
        self.transform = transform

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows.iloc[index]

        with Image.open(
            self.root / str(row["rel_path"])
        ) as image:
            tensor = self.transform(image)

        return tensor, int(row["class_id"])


SEED = 2026
BATCH_SIZE = 30

d2_train_rows = d2_manifest[
    d2_manifest["split"] == "train"
].copy()

d2_val_rows = d2_manifest[
    d2_manifest["split"] == "val"
].copy()

assert len(d2_train_rows) == 3327
assert len(d2_val_rows) == 712

assert set(d2_train_rows["class_id"]) == set(range(4))
assert set(d2_val_rows["class_id"]) == set(range(4))

d2_train_dataset = Paper6RiceDataset(
    d2_train_rows,
    D2_ROOT,
    Paper6Transform(training=True, seed=SEED)
)

d2_val_dataset = Paper6RiceDataset(
    d2_val_rows,
    D2_ROOT,
    Paper6Transform(training=False)
)

audit_generator = torch.Generator().manual_seed(SEED)

d2_train_loader = DataLoader(
    d2_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=audit_generator,
    num_workers=0,
    drop_last=False
)

d2_val_loader = DataLoader(
    d2_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    drop_last=False
)

assert len(d2_train_loader) == 111
assert len(d2_val_loader) == 24

assert isinstance(
    d2_train_loader.sampler,
    RandomSampler
)

assert isinstance(
    d2_val_loader.sampler,
    SequentialSampler
)

train_x, train_y = next(iter(d2_train_loader))
val_x, val_y = next(iter(d2_val_loader))

assert train_x.shape == (30, 3, 227, 227)
assert val_x.shape == (30, 3, 227, 227)

assert train_y.dtype == torch.int64
assert val_y.dtype == torch.int64

assert torch.isfinite(train_x).all()
assert torch.isfinite(val_x).all()

assert d2_train_dataset.transform.training
assert not d2_val_dataset.transform.training

val_first, _ = d2_val_dataset[0]
val_second, _ = d2_val_dataset[0]

assert torch.equal(val_first, val_second)

print("Dataset: D2")
print("Training images:", len(d2_train_dataset))
print("Validation images:", len(d2_val_dataset))
print("Test images:", 713)

print("\nTraining batches:", len(d2_train_loader))
print("Validation batches:", len(d2_val_loader))

print("Training batch shape:", tuple(train_x.shape))
print("Validation batch shape:", tuple(val_x.shape))

print(
    "Training shuffle:",
    isinstance(d2_train_loader.sampler, RandomSampler)
)

print(
    "Training augmentation:",
    d2_train_dataset.transform.training
)

print(
    "Validation augmentation:",
    d2_val_dataset.transform.training
)

print(
    "Validation deterministic:",
    torch.equal(val_first, val_second)
)

print("\nPAPER 6 D2 PREPROCESSING PIPELINE: PASSED")


Dataset: D2
Training images: 3327
Validation images: 712
Test images: 713

Training batches: 111
Validation batches: 24
Training batch shape: (30, 3, 227, 227)
Validation batch shape: (30, 3, 227, 227)
Training shuffle: True
Training augmentation: True
Validation augmentation: False
Validation deterministic: True

PAPER 6 D2 PREPROCESSING PIPELINE: PASSED


In [5]:

import json
import hashlib
import torch
import torchvision
import cv2

from pathlib import Path

SEED = 2026
BATCH_SIZE = 30
EPOCHS = 10

BASE_LR = 0.0003
NEW_LAYER_FACTOR = 10.0
MOMENTUM = 0.9
WEIGHT_DECAY = 0.0001


def build_paper6_optimizer(net):
    groups = [
        {
            "params": [
                p for n, p in net.features.named_parameters()
                if p.requires_grad and n.endswith("weight")
            ],
            "lr": BASE_LR,
            "weight_decay": WEIGHT_DECAY,
            "group_name": "backbone_weights"
        },
        {
            "params": [
                p for n, p in net.features.named_parameters()
                if p.requires_grad and n.endswith("bias")
            ],
            "lr": BASE_LR,
            "weight_decay": 0.0,
            "group_name": "backbone_biases"
        }
    ]

    for name in (
        "mfa_1x1",
        "mfa_3x3",
        "class_projection"
    ):
        layer = getattr(net, name)

        groups.append({
            "params": [layer.weight],
            "lr": BASE_LR * NEW_LAYER_FACTOR,
            "weight_decay": WEIGHT_DECAY,
            "group_name": name + "_weight"
        })

        groups.append({
            "params": [layer.bias],
            "lr": BASE_LR * NEW_LAYER_FACTOR,
            "weight_decay": 0.0,
            "group_name": name + "_bias"
        })

    optimizer = torch.optim.SGD(
        groups,
        lr=BASE_LR,
        momentum=MOMENTUM,
        nesterov=False
    )

    assigned = [
        p
        for group in optimizer.param_groups
        for p in group["params"]
    ]

    trainable = [
        p for p in net.parameters()
        if p.requires_grad
    ]

    assigned_ids = [id(p) for p in assigned]
    trainable_ids = [id(p) for p in trainable]

    assert len(optimizer.param_groups) == 8
    assert len(assigned_ids) == len(set(assigned_ids))
    assert set(assigned_ids) == set(trainable_ids)

    assert sum(p.numel() for p in assigned) == 722518

    assert sum(
        p.numel()
        for p in net.parameters()
        if not p.requires_grad
    ) == 25632

    assert optimizer.defaults["momentum"] == MOMENTUM

    for group in optimizer.param_groups:
        is_backbone = group["group_name"].startswith(
            "backbone"
        )

        expected_lr = (
            BASE_LR
            if is_backbone
            else BASE_LR * NEW_LAYER_FACTOR
        )

        expected_decay = (
            0.0
            if group["group_name"].endswith("biases")
            or group["group_name"].endswith("_bias")
            else WEIGHT_DECAY
        )

        assert abs(group["lr"] - expected_lr) < 1e-12
        assert group["weight_decay"] == expected_decay

    return optimizer


optimizer = build_paper6_optimizer(model)

assert model.class_projection.out_channels == 4

OUTPUT_DIR = Path(
    "/kaggle/working/Paper6_MFA_SqueezeNet_D2_Revised"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

manifest_sha256 = hashlib.sha256(
    d2_manifest_path.read_bytes()
).hexdigest()

configuration = {
    "paper": "Upadhyay and Dwivedi (2025), DOI: 10.1111/jph.70092",
    "implementation": "Independent PyTorch adaptation, not exact MATLAB reproduction",
    "dataset": "D2, four classes",
    "manifest_sha256": manifest_sha256,
    "split_counts": {
        "train": 3327,
        "val": 712,
        "test": 713
    },
    "class_mapping": d2_class_map,
    "seed": SEED,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "input_size": [227, 227],
    "backbone": "ImageNet-pretrained SqueezeNet 1.1",
    "frozen_feature_indices": [0, 3, 4],
    "mfa_branch_filters": [5, 5],
    "class_projection": "10-to-4 pointwise convolution",
    "preprocessing": "RGB resize, Lab L* min-max stretch, ImageNet normalization",
    "augmentation": "Horizontal reflection, +/-30-pixel translations, independent 0.9-1.1 scaling",
    "optimizer": {
        "name": "SGD",
        "backbone_lr": BASE_LR,
        "new_layer_lr": BASE_LR * NEW_LAYER_FACTOR,
        "momentum": MOMENTUM,
        "weight_decay_weights": WEIGHT_DECAY,
        "weight_decay_biases": 0.0
    },
    "software": {
        "pytorch": torch.__version__,
        "torchvision": torchvision.__version__,
        "opencv": cv2.__version__
    },
    "disclosures": [
        "The original paper used five disease classes.",
        "The final 10-to-4 projection is an independent reconstruction choice.",
        "The new-layer learning-rate factor was not specified numerically in the paper.",
        "Training uses the verified D2 duplicate-aware split manifest.",
        "The original paper used six epochs; this experiment uses ten.",
        "The same architecture and training settings are used for D1 and D2 except for the number of output classes."
    ]
}

config_path = OUTPUT_DIR / "pretraining_config.json"

config_path.write_text(
    json.dumps(
        configuration,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print("Optimizer:", type(optimizer).__name__)
print("Momentum:", optimizer.defaults["momentum"])
print("Parameter groups:", len(optimizer.param_groups))

for group in optimizer.param_groups:
    count = sum(
        p.numel()
        for p in group["params"]
    )

    print(
        group["group_name"],
        "| Parameters:", count,
        "| LR:", group["lr"],
        "| Weight decay:", group["weight_decay"]
    )

print("\nTrainable parameters:", sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
))

print("Frozen parameters:", sum(
    p.numel()
    for p in model.parameters()
    if not p.requires_grad
))

print("Configuration saved:", config_path)

print("\nPAPER 6 D2 OPTIMIZER CONFIGURATION: PASSED")


Optimizer: SGD
Momentum: 0.9
Parameter groups: 8
backbone_weights | Parameters: 694272 | LR: 0.0003 | Weight decay: 0.0001
backbone_biases | Parameters: 2592 | LR: 0.0003 | Weight decay: 0.0
mfa_1x1_weight | Parameters: 2560 | LR: 0.0029999999999999996 | Weight decay: 0.0001
mfa_1x1_bias | Parameters: 5 | LR: 0.0029999999999999996 | Weight decay: 0.0
mfa_3x3_weight | Parameters: 23040 | LR: 0.0029999999999999996 | Weight decay: 0.0001
mfa_3x3_bias | Parameters: 5 | LR: 0.0029999999999999996 | Weight decay: 0.0
class_projection_weight | Parameters: 40 | LR: 0.0029999999999999996 | Weight decay: 0.0001
class_projection_bias | Parameters: 4 | LR: 0.0029999999999999996 | Weight decay: 0.0

Trainable parameters: 722518
Frozen parameters: 25632
Configuration saved: /kaggle/working/Paper6_MFA_SqueezeNet_D2_Revised/pretraining_config.json

PAPER 6 D2 OPTIMIZER CONFIGURATION: PASSED


In [6]:

import json
import hashlib
import random
from pathlib import Path

import torch
import torch.nn as nn

assert torch.cuda.is_available(), "Enable Kaggle GPU first."

required_names = [
    "Paper6MFASqueezeNet",
    "build_paper6_optimizer",
    "train_x",
    "train_y",
    "val_x",
    "OUTPUT_DIR",
    "d2_manifest_path"
]

missing_names = [
    name for name in required_names
    if name not in globals()
]

assert not missing_names, f"Missing definitions: {missing_names}"

device = torch.device("cuda:0")
seed = 2026

random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

output_dir = Path(OUTPUT_DIR)

config_path = output_dir / "pretraining_config.json"

assert config_path.is_file()

config = json.loads(
    config_path.read_text(encoding="utf-8")
)

manifest_hash = hashlib.sha256(
    d2_manifest_path.read_bytes()
).hexdigest()

assert config["manifest_sha256"] == manifest_hash
assert config["epochs"] == 10
assert config["batch_size"] == 30
assert config["class_projection"] == "10-to-4 pointwise convolution"

smoke_model = Paper6MFASqueezeNet(
    num_classes=4
).to(device)

smoke_optimizer = build_paper6_optimizer(
    smoke_model
)

assert smoke_model.class_projection.out_channels == 4
assert len(smoke_optimizer.param_groups) == 8

assert sum(
    p.numel()
    for p in smoke_model.parameters()
    if p.requires_grad
) == 722518

assert sum(
    p.numel()
    for p in smoke_model.parameters()
    if not p.requires_grad
) == 25632

inputs = train_x.to(device)
targets = train_y.to(device)

assert inputs.shape == (30, 3, 227, 227)
assert targets.shape == (30,)
assert targets.dtype == torch.int64
assert bool(torch.all((targets >= 0) & (targets < 4)))

frozen_before = {
    name: parameter.detach().clone()
    for name, parameter in smoke_model.named_parameters()
    if not parameter.requires_grad
}

classifier_before = (
    smoke_model.class_projection.weight
    .detach().clone()
)

smoke_model.train()
smoke_optimizer.zero_grad(set_to_none=True)

logits = smoke_model(inputs)

assert logits.shape == (30, 4)
assert torch.isfinite(logits).all().item()

loss = nn.CrossEntropyLoss()(
    logits,
    targets
)

assert torch.isfinite(loss).item()

loss.backward()

assert all(
    parameter.grad is None
    for parameter in smoke_model.parameters()
    if not parameter.requires_grad
)

assert all(
    parameter.grad is not None
    and torch.isfinite(parameter.grad).all().item()
    for parameter in smoke_model.parameters()
    if parameter.requires_grad
)

smoke_optimizer.step()

frozen_unchanged = all(
    torch.equal(
        frozen_before[name],
        parameter.detach()
    )
    for name, parameter in smoke_model.named_parameters()
    if not parameter.requires_grad
)

classifier_updated = not torch.equal(
    classifier_before,
    smoke_model.class_projection.weight.detach()
)

assert frozen_unchanged
assert classifier_updated

smoke_model.eval()

with torch.inference_mode():
    validation_logits = smoke_model(
        val_x.to(device)
    )

assert validation_logits.shape == (30, 4)
assert torch.isfinite(validation_logits).all().item()

checkpoint_path = output_dir / "d2_gpu_smoke_checkpoint.pth"

torch.save(
    {
        "epoch": 0,
        "model_state_dict": smoke_model.state_dict(),
        "optimizer_state_dict": smoke_optimizer.state_dict(),
        "num_classes": 4
    },
    checkpoint_path
)

restored = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

assert restored["epoch"] == 0
assert restored["num_classes"] == 4

assert torch.equal(
    restored["model_state_dict"]["class_projection.weight"],
    smoke_model.class_projection.weight.detach().cpu()
)

checkpoint_path.unlink()

report = {
    "status": "PASSED",
    "dataset": "D2",
    "purpose": "GPU smoke test, not full training",
    "gpu": torch.cuda.get_device_name(0),
    "batch_size": int(inputs.shape[0]),
    "num_classes": 4,
    "training_loss": float(loss.detach().cpu().item()),
    "frozen_weights_unchanged": frozen_unchanged,
    "classifier_weights_updated": classifier_updated,
    "validation_forward_passed": True,
    "checkpoint_roundtrip_passed": True
}

report_path = output_dir / "gpu_smoke_verification.json"

report_path.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

print("GPU:", report["gpu"])
print("Training batch:", tuple(inputs.shape))
print("Training loss:", round(report["training_loss"], 6))
print("Classification outputs:", logits.shape[1])
print("Frozen weights unchanged:", frozen_unchanged)
print("Classifier weights updated:", classifier_updated)
print("Validation forward pass: PASSED")
print("Checkpoint save/load: PASSED")
print("Verification report:", report_path)

print("\nPAPER 6 D2 GPU SMOKE TEST: PASSED")


GPU: Tesla T4
Training batch: (30, 3, 227, 227)
Training loss: 1.6922
Classification outputs: 4
Frozen weights unchanged: True
Classifier weights updated: True
Validation forward pass: PASSED
Checkpoint save/load: PASSED
Verification report: /kaggle/working/Paper6_MFA_SqueezeNet_D2_Revised/gpu_smoke_verification.json

PAPER 6 D2 GPU SMOKE TEST: PASSED


In [7]:

import json
import hashlib
import random
from pathlib import Path

import torch
import torch.nn as nn

assert torch.cuda.is_available(), "Enable Kaggle GPU first."

required_names = [
    "Paper6MFASqueezeNet",
    "build_paper6_optimizer",
    "train_x",
    "train_y",
    "val_x",
    "OUTPUT_DIR",
    "d2_manifest_path"
]

missing_names = [
    name for name in required_names
    if name not in globals()
]

assert not missing_names, f"Missing definitions: {missing_names}"

device = torch.device("cuda:0")
seed = 2026

random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

output_dir = Path(OUTPUT_DIR)

config_path = output_dir / "pretraining_config.json"

assert config_path.is_file()

config = json.loads(
    config_path.read_text(encoding="utf-8")
)

manifest_hash = hashlib.sha256(
    d2_manifest_path.read_bytes()
).hexdigest()

assert config["manifest_sha256"] == manifest_hash
assert config["epochs"] == 10
assert config["batch_size"] == 30
assert config["class_projection"] == "10-to-4 pointwise convolution"

smoke_model = Paper6MFASqueezeNet(
    num_classes=4
).to(device)

smoke_optimizer = build_paper6_optimizer(
    smoke_model
)

assert smoke_model.class_projection.out_channels == 4
assert len(smoke_optimizer.param_groups) == 8

assert sum(
    p.numel()
    for p in smoke_model.parameters()
    if p.requires_grad
) == 722518

assert sum(
    p.numel()
    for p in smoke_model.parameters()
    if not p.requires_grad
) == 25632

inputs = train_x.to(device)
targets = train_y.to(device)

assert inputs.shape == (30, 3, 227, 227)
assert targets.shape == (30,)
assert targets.dtype == torch.int64
assert bool(torch.all((targets >= 0) & (targets < 4)))

frozen_before = {
    name: parameter.detach().clone()
    for name, parameter in smoke_model.named_parameters()
    if not parameter.requires_grad
}

classifier_before = (
    smoke_model.class_projection.weight
    .detach().clone()
)

smoke_model.train()
smoke_optimizer.zero_grad(set_to_none=True)

logits = smoke_model(inputs)

assert logits.shape == (30, 4)
assert torch.isfinite(logits).all().item()

loss = nn.CrossEntropyLoss()(
    logits,
    targets
)

assert torch.isfinite(loss).item()

loss.backward()

assert all(
    parameter.grad is None
    for parameter in smoke_model.parameters()
    if not parameter.requires_grad
)

assert all(
    parameter.grad is not None
    and torch.isfinite(parameter.grad).all().item()
    for parameter in smoke_model.parameters()
    if parameter.requires_grad
)

smoke_optimizer.step()

frozen_unchanged = all(
    torch.equal(
        frozen_before[name],
        parameter.detach()
    )
    for name, parameter in smoke_model.named_parameters()
    if not parameter.requires_grad
)

classifier_updated = not torch.equal(
    classifier_before,
    smoke_model.class_projection.weight.detach()
)

assert frozen_unchanged
assert classifier_updated

smoke_model.eval()

with torch.inference_mode():
    validation_logits = smoke_model(
        val_x.to(device)
    )

assert validation_logits.shape == (30, 4)
assert torch.isfinite(validation_logits).all().item()

checkpoint_path = output_dir / "d2_gpu_smoke_checkpoint.pth"

torch.save(
    {
        "epoch": 0,
        "model_state_dict": smoke_model.state_dict(),
        "optimizer_state_dict": smoke_optimizer.state_dict(),
        "num_classes": 4
    },
    checkpoint_path
)

restored = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

assert restored["epoch"] == 0
assert restored["num_classes"] == 4

assert torch.equal(
    restored["model_state_dict"]["class_projection.weight"],
    smoke_model.class_projection.weight.detach().cpu()
)

checkpoint_path.unlink()

report = {
    "status": "PASSED",
    "dataset": "D2",
    "purpose": "GPU smoke test, not full training",
    "gpu": torch.cuda.get_device_name(0),
    "batch_size": int(inputs.shape[0]),
    "num_classes": 4,
    "training_loss": float(loss.detach().cpu().item()),
    "frozen_weights_unchanged": frozen_unchanged,
    "classifier_weights_updated": classifier_updated,
    "validation_forward_passed": True,
    "checkpoint_roundtrip_passed": True
}

report_path = output_dir / "gpu_smoke_verification.json"

report_path.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

print("GPU:", report["gpu"])
print("Training batch:", tuple(inputs.shape))
print("Training loss:", round(report["training_loss"], 6))
print("Classification outputs:", logits.shape[1])
print("Frozen weights unchanged:", frozen_unchanged)
print("Classifier weights updated:", classifier_updated)
print("Validation forward pass: PASSED")
print("Checkpoint save/load: PASSED")
print("Verification report:", report_path)

print("\nPAPER 6 D2 GPU SMOKE TEST: PASSED")


GPU: Tesla T4
Training batch: (30, 3, 227, 227)
Training loss: 1.6922
Classification outputs: 4
Frozen weights unchanged: True
Classifier weights updated: True
Validation forward pass: PASSED
Checkpoint save/load: PASSED
Verification report: /kaggle/working/Paper6_MFA_SqueezeNet_D2_Revised/gpu_smoke_verification.json

PAPER 6 D2 GPU SMOKE TEST: PASSED


In [8]:

import csv
import hashlib
import json
import os
import random
import time
import zipfile
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

assert torch.cuda.is_available(), "Kaggle GPU is required."

SEED = 2026
EPOCHS = 10
BATCH_SIZE = 30
DEVICE = torch.device("cuda:0")

output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

config_file = output_dir / "pretraining_config.json"
last_file = output_dir / "last_checkpoint.pth"
best_file = output_dir / "best_model.pth"
history_file = output_dir / "training_history.csv"
archive_file = output_dir / "training_backup.zip"

config = json.loads(
    config_file.read_text(encoding="utf-8")
)

manifest_hash = hashlib.sha256(
    d2_manifest_path.read_bytes()
).hexdigest()

assert config["manifest_sha256"] == manifest_hash
assert config["epochs"] == EPOCHS
assert config["batch_size"] == BATCH_SIZE
assert config["class_projection"] == "10-to-4 pointwise convolution"
assert config["optimizer"]["backbone_lr"] == 0.0003
assert abs(config["optimizer"]["new_layer_lr"] - 0.003) < 1e-12

assert d2_manifest["split"].value_counts().to_dict() == {
    "train": 3327,
    "val": 712,
    "test": 713
}

if not last_file.exists() and (
    history_file.exists() or best_file.exists()
):
    raise RuntimeError(
        "Previous training artifacts exist without a "
        "resumable checkpoint. Preserve them before proceeding."
    )

random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

training_rows = d2_manifest.loc[
    d2_manifest["split"] == "train"
].copy()

validation_rows = d2_manifest.loc[
    d2_manifest["split"] == "val"
].copy()

training_dataset = Paper6RiceDataset(
    training_rows,
    D2_ROOT,
    Paper6Transform(training=True, seed=SEED)
)

validation_dataset = Paper6RiceDataset(
    validation_rows,
    D2_ROOT,
    Paper6Transform(training=False)
)

shuffle_generator = torch.Generator().manual_seed(SEED)

training_loader = DataLoader(
    training_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=shuffle_generator,
    num_workers=0,
    drop_last=False
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    drop_last=False
)

assert len(training_dataset) == 3327
assert len(validation_dataset) == 712
assert len(training_loader) == 111
assert len(validation_loader) == 24

training_model = Paper6MFASqueezeNet(
    num_classes=4
).to(DEVICE)

training_optimizer = build_paper6_optimizer(
    training_model
)

criterion = nn.CrossEntropyLoss()

assert sum(
    p.numel()
    for p in training_model.parameters()
    if p.requires_grad
) == 722518

assert sum(
    p.numel()
    for p in training_model.parameters()
    if not p.requires_grad
) == 25632

assert len(training_optimizer.param_groups) == 8


def atomic_torch_save(value, path):
    temporary = path.with_name(path.name + ".tmp")
    torch.save(value, temporary)
    os.replace(temporary, path)


def atomic_history_save(rows, path):
    temporary = path.with_name(path.name + ".tmp")

    with temporary.open(
        "w", newline="", encoding="utf-8"
    ) as handle:
        writer = csv.DictWriter(
            handle,
            fieldnames=[
                "epoch",
                "train_loss",
                "train_accuracy",
                "val_loss",
                "val_accuracy",
                "seconds"
            ]
        )

        writer.writeheader()
        writer.writerows(rows)

    os.replace(temporary, path)


def atomic_backup(path):
    temporary = path.with_name(path.name + ".tmp")

    with zipfile.ZipFile(
        temporary,
        "w",
        compression=zipfile.ZIP_DEFLATED,
        compresslevel=3
    ) as archive:
        for item in (
            config_file,
            history_file,
            last_file,
            best_file
        ):
            if item.exists():
                archive.write(
                    item,
                    arcname=item.name
                )

    os.replace(temporary, path)


start_epoch = 1
best_loss = float("inf")
best_epoch = None
history = []

if last_file.exists():
    checkpoint = torch.load(
        last_file,
        map_location=DEVICE,
        weights_only=True
    )

    assert checkpoint["manifest_sha256"] == manifest_hash
    assert checkpoint["seed"] == SEED
    assert checkpoint["batch_size"] == BATCH_SIZE
    assert checkpoint["num_classes"] == 4

    training_model.load_state_dict(
        checkpoint["model_state_dict"],
        strict=True
    )

    training_optimizer.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )

    shuffle_generator.set_state(
        checkpoint["shuffle_generator_state"]
    )

    training_dataset.transform.rng.setstate(
        checkpoint["augmentation_rng_state"]
    )

    torch.set_rng_state(
        checkpoint["torch_cpu_rng_state"].cpu()
    )

    torch.cuda.set_rng_state_all(
        checkpoint["torch_cuda_rng_states"]
    )

    history = checkpoint["history"]
    best_loss = float(checkpoint["best_val_loss"])
    best_epoch = int(checkpoint["best_epoch"])
    start_epoch = int(checkpoint["epoch"]) + 1

    assert len(history) == start_epoch - 1
    assert best_file.exists()

    print(
        "RESUMING FROM COMPLETED EPOCH:",
        start_epoch - 1,
        flush=True
    )

else:
    print(
        "STARTING FRESH D2 TRAINING FROM IMAGENET WEIGHTS",
        flush=True
    )

print("GPU:", torch.cuda.get_device_name(0), flush=True)
print(
    "Training images:", len(training_dataset),
    "| Validation images:", len(validation_dataset),
    flush=True
)
print("Test set: NOT USED", flush=True)


for epoch in range(start_epoch, EPOCHS + 1):
    start_time = time.perf_counter()

    training_model.train()

    train_loss_sum = 0.0
    train_correct = 0
    train_count = 0

    for images, targets in training_loader:
        images = images.to(DEVICE)
        targets = targets.to(DEVICE)

        training_optimizer.zero_grad(
            set_to_none=True
        )

        outputs = training_model(images)
        loss = criterion(outputs, targets)

        if not torch.isfinite(loss).item():
            raise FloatingPointError(
                f"Nonfinite training loss at epoch {epoch}."
            )

        loss.backward()
        training_optimizer.step()

        count = targets.size(0)

        train_loss_sum += (
            float(loss.detach().item()) * count
        )

        train_correct += int(
            (
                outputs.detach().argmax(dim=1)
                == targets
            ).sum().item()
        )

        train_count += count

    training_model.eval()

    val_loss_sum = 0.0
    val_correct = 0
    val_count = 0

    with torch.inference_mode():
        for images, targets in validation_loader:
            images = images.to(DEVICE)
            targets = targets.to(DEVICE)

            outputs = training_model(images)
            loss = criterion(outputs, targets)

            if not torch.isfinite(loss).item():
                raise FloatingPointError(
                    f"Nonfinite validation loss at epoch {epoch}."
                )

            count = targets.size(0)

            val_loss_sum += (
                float(loss.item()) * count
            )

            val_correct += int(
                (
                    outputs.argmax(dim=1)
                    == targets
                ).sum().item()
            )

            val_count += count

    assert train_count == 3327
    assert val_count == 712

    train_loss = train_loss_sum / train_count
    val_loss = val_loss_sum / val_count

    train_accuracy = train_correct / train_count
    val_accuracy = val_correct / val_count

    elapsed = time.perf_counter() - start_time

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "val_loss": val_loss,
        "val_accuracy": val_accuracy,
        "seconds": elapsed
    })

    if val_loss < best_loss:
        best_loss = val_loss
        best_epoch = epoch

        atomic_torch_save(
            {
                "epoch": epoch,
                "model_state_dict": training_model.state_dict(),
                "val_loss": val_loss,
                "val_accuracy": val_accuracy,
                "manifest_sha256": manifest_hash,
                "num_classes": 4
            },
            best_file
        )

    atomic_torch_save(
        {
            "epoch": epoch,
            "model_state_dict": training_model.state_dict(),
            "optimizer_state_dict": training_optimizer.state_dict(),
            "best_epoch": best_epoch,
            "best_val_loss": best_loss,
            "history": history,
            "seed": SEED,
            "batch_size": BATCH_SIZE,
            "num_classes": 4,
            "manifest_sha256": manifest_hash,
            "shuffle_generator_state": shuffle_generator.get_state(),
            "augmentation_rng_state": training_dataset.transform.rng.getstate(),
            "torch_cpu_rng_state": torch.get_rng_state(),
            "torch_cuda_rng_states": torch.cuda.get_rng_state_all()
        },
        last_file
    )

    atomic_history_save(
        history,
        history_file
    )

    atomic_backup(
        archive_file
    )

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train loss {train_loss:.4f}, "
        f"acc {train_accuracy:.4f} | "
        f"Val loss {val_loss:.4f}, "
        f"acc {val_accuracy:.4f} | "
        f"{elapsed:.1f}s | "
        f"best epoch {best_epoch} | saved",
        flush=True
    )

print("\nCOMPLETED EPOCHS:", len(history))
print("BEST VALIDATION EPOCH:", best_epoch)
print("BEST VALIDATION LOSS:", round(best_loss, 6))
print("HISTORY:", history_file)
print("BEST MODEL:", best_file)
print("RESUME CHECKPOINT:", last_file)
print("BACKUP ARCHIVE:", archive_file)

print("\nPAPER 6 D2 REVISED TRAINING: COMPLETE")


STARTING FRESH D2 TRAINING FROM IMAGENET WEIGHTS
GPU: Tesla T4
Training images: 3327 | Validation images: 712
Test set: NOT USED
Epoch 01/10 | Train loss 0.3607, acc 0.8617 | Val loss 0.3087, acc 0.8904 | 331.4s | best epoch 1 | saved
Epoch 02/10 | Train loss 0.1054, acc 0.9672 | Val loss 0.1248, acc 0.9551 | 37.3s | best epoch 2 | saved
Epoch 03/10 | Train loss 0.0460, acc 0.9853 | Val loss 0.0735, acc 0.9719 | 37.8s | best epoch 3 | saved
Epoch 04/10 | Train loss 0.0280, acc 0.9901 | Val loss 0.0467, acc 0.9803 | 37.5s | best epoch 4 | saved
Epoch 05/10 | Train loss 0.0223, acc 0.9928 | Val loss 0.0523, acc 0.9817 | 37.7s | best epoch 4 | saved
Epoch 06/10 | Train loss 0.0124, acc 0.9958 | Val loss 0.1008, acc 0.9705 | 37.4s | best epoch 4 | saved
Epoch 07/10 | Train loss 0.0202, acc 0.9925 | Val loss 0.0341, acc 0.9860 | 37.8s | best epoch 7 | saved
Epoch 08/10 | Train loss 0.0120, acc 0.9967 | Val loss 0.0836, acc 0.9761 | 37.7s | best epoch 7 | saved
Epoch 09/10 | Train loss 0.010

In [9]:

import csv
import json
import math
import hashlib
import zipfile
from pathlib import Path

import pandas as pd
import torch

from IPython.display import FileLink, display

root = Path(
    "/kaggle/working/Paper6_MFA_SqueezeNet_D2_Revised"
)

split_root = Path(
    "/kaggle/input/datasets/nayebul0/"
    "assignment02-verified-train-val-test-splits"
)

manifest_path = split_root / "D2_split_manifest.csv"

required = [
    "pretraining_config.json",
    "training_history.csv",
    "best_model.pth",
    "last_checkpoint.pth",
    "training_backup.zip",
    "gpu_smoke_verification.json"
]

for filename in required:
    path = root / filename
    assert path.is_file(), f"Missing file: {filename}"
    assert path.stat().st_size > 0

assert manifest_path.is_file()

config = json.loads(
    (root / "pretraining_config.json").read_text(
        encoding="utf-8"
    )
)

manifest_hash = hashlib.sha256(
    manifest_path.read_bytes()
).hexdigest()

assert config["manifest_sha256"] == manifest_hash
assert config["epochs"] == 10
assert config["batch_size"] == 30

manifest_check = pd.read_csv(manifest_path)

assert manifest_check["split"].value_counts().to_dict() == {
    "train": 3327,
    "val": 712,
    "test": 713
}

with (root / "training_history.csv").open(
    newline="",
    encoding="utf-8"
) as handle:
    history = list(csv.DictReader(handle))

assert len(history) == 10
assert [int(row["epoch"]) for row in history] == list(range(1, 11))

for row in history:
    for field in (
        "train_loss",
        "train_accuracy",
        "val_loss",
        "val_accuracy"
    ):
        assert math.isfinite(float(row[field]))

    assert 0 <= float(row["train_accuracy"]) <= 1
    assert 0 <= float(row["val_accuracy"]) <= 1

best_row = min(
    history,
    key=lambda row: float(row["val_loss"])
)

assert int(best_row["epoch"]) == 10
assert math.isclose(
    float(best_row["val_loss"]),
    0.007863,
    abs_tol=0.000001
)

assert math.isclose(
    float(history[-1]["val_accuracy"]),
    710 / 712,
    abs_tol=1e-12
)

best_checkpoint = torch.load(
    root / "best_model.pth",
    map_location="cpu",
    weights_only=True
)

last_checkpoint = torch.load(
    root / "last_checkpoint.pth",
    map_location="cpu",
    weights_only=True
)

assert best_checkpoint["epoch"] == 10
assert best_checkpoint["num_classes"] == 4
assert last_checkpoint["epoch"] == 10
assert last_checkpoint["num_classes"] == 4
assert last_checkpoint["best_epoch"] == 10

assert best_checkpoint["manifest_sha256"] == manifest_hash
assert last_checkpoint["manifest_sha256"] == manifest_hash

assert math.isclose(
    float(best_checkpoint["val_loss"]),
    float(best_row["val_loss"]),
    abs_tol=1e-12
)

assert tuple(
    best_checkpoint["model_state_dict"][
        "class_projection.weight"
    ].shape
) == (4, 10, 1, 1)

archive_path = root / "training_backup.zip"

archive_members = [
    "pretraining_config.json",
    "training_history.csv",
    "best_model.pth",
    "last_checkpoint.pth"
]

with zipfile.ZipFile(archive_path) as archive:
    assert archive.testzip() is None

    for filename in archive_members:
        assert filename in archive.namelist()

        saved_hash = hashlib.sha256(
            (root / filename).read_bytes()
        ).hexdigest()

        zipped_hash = hashlib.sha256(
            archive.read(filename)
        ).hexdigest()

        assert saved_hash == zipped_hash

archive_hash = hashlib.sha256(
    archive_path.read_bytes()
).hexdigest()

print("Training epochs:", len(history))
print("Best epoch:", best_checkpoint["epoch"])
print("Best validation loss:", best_row["val_loss"])
print(
    "Final validation accuracy:",
    round(float(history[-1]["val_accuracy"]), 6)
)
print("Last checkpoint epoch:", last_checkpoint["epoch"])
print("D2 manifest fingerprint: PASSED")
print("Backup integrity: PASSED")
print("Archive matches saved files: True")
print("Archive SHA256:", archive_hash)
print(
    "Archive size (MB):",
    round(archive_path.stat().st_size / 1048576, 2)
)

print("\nPAPER 6 D2 TRAINING ARTIFACT AUDIT: PASSED")

display(FileLink(str(archive_path)))


Training epochs: 10
Best epoch: 10
Best validation loss: 0.007862830726652878
Final validation accuracy: 0.997191
Last checkpoint epoch: 10
D2 manifest fingerprint: PASSED
Backup integrity: PASSED
Archive matches saved files: True
Archive SHA256: 37d84a69d14da311ed3066d7d549fe101b65265735597f3cb0331c0525257f1e
Archive size (MB): 7.94

PAPER 6 D2 TRAINING ARTIFACT AUDIT: PASSED


/kaggle/working/Paper6_MFA_SqueezeNet_D2_Revised/training_backup.zip

In [10]:

import json
import hashlib
import pandas as pd
from pathlib import Path

split_root = Path(
    "/kaggle/input/datasets/nayebul0/"
    "assignment02-verified-train-val-test-splits"
)

manifest_path = split_root / "D2_split_manifest.csv"
gate_path = split_root / "gate_status.json"
class_map_path = split_root / "class_maps.json"

assert manifest_path.is_file()
assert gate_path.is_file()
assert class_map_path.is_file()

df = pd.read_csv(manifest_path)

gate = json.loads(
    gate_path.read_text(encoding="utf-8")
)

class_maps = json.loads(
    class_map_path.read_text(encoding="utf-8")
)

expected_counts = {
    "train": 3327,
    "val": 712,
    "test": 713
}

assert len(df) == 4752
assert df["split"].value_counts().to_dict() == expected_counts
assert df["rel_path"].is_unique
assert df["rel_path"].notna().all()
assert df["class_id"].between(0, 3).all()

assert class_maps["D2"] == {
    "Bacterial Blight": 0,
    "Blast": 1,
    "Brown Spot": 2,
    "Tungro": 3
}

manifest_hash = hashlib.sha256(
    manifest_path.read_bytes()
).hexdigest()

config_path = Path(
    "/kaggle/working/Paper6_MFA_SqueezeNet_D2_Revised/"
    "pretraining_config.json"
)

config = json.loads(
    config_path.read_text(encoding="utf-8")
)

assert config["manifest_sha256"] == manifest_hash

print("Gate status:", gate.get("status"))
print("Gate record keys:", list(gate.keys()))
print("Manifest SHA256 matches training: True")
print("Duplicate file paths across splits: 0")
print("Split counts:", expected_counts)

print("\nD2 CLASS DISTRIBUTION")
print(pd.crosstab(df["class_id"], df["split"]))

group_columns = [
    column for column in df.columns
    if column.lower() in {
        "group_id",
        "cluster_id",
        "duplicate_group_id",
        "near_duplicate_group_id",
        "similarity_group_id",
        "split_group_id"
    }
]

print("\nGroup identifier columns:", group_columns)

for column in group_columns:
    grouped = df.dropna(
        subset=[column]
    ).groupby(column)["split"].nunique()

    overlaps = int((grouped > 1).sum())

    print(
        f"{column}: {overlaps} groups span multiple splits"
    )

    assert overlaps == 0

assert gate.get("status") == "PASS_SPLITS_FROZEN"

print("\nRecorded split gate: PASSED")
print("Dataset identity and partition counts: PASSED")

print("\nPAPER 6 D2 SPLIT-INTEGRITY AUDIT: PASSED")


Gate status: PASS_SPLITS_FROZEN
Gate record keys: ['status', 'source_urls', 'actual_roots', 'reported_source_counts', 'expected_checked_counts', 'observed_image_counts', 'observed_class_counts', 'd1_skipped_augmented_files', 'scan_errors', 'exact_duplicate_pairs', 'near_duplicate_pairs', 'documented_manual_exclusions', 'auto_deduplicated_images', 'grouping_policy', 'grouping_counts', 'stale_review_pairs_after_exclusions', 'seed', 'near_hamming_radius', 'review_csv_path', 'blockers', 'splits', 'warning']
Manifest SHA256 matches training: True
Duplicate file paths across splits: 0
Split counts: {'train': 3327, 'val': 712, 'test': 713}

D2 CLASS DISTRIBUTION
split     test  train  val
class_id                  
0          193    899  192
1          144    672  144
2          180    840  180
3          196    916  196

Group identifier columns: []

Recorded split gate: PASSED
Dataset identity and partition counts: PASSED

PAPER 6 D2 SPLIT-INTEGRITY AUDIT: PASSED


In [11]:

import json
import hashlib
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    ConfusionMatrixDisplay
)

device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

result_dir = Path(OUTPUT_DIR) / "evaluation"
result_dir.mkdir(parents=True, exist_ok=True)

metrics_path = result_dir / "test_metrics.json"

assert not metrics_path.exists(), (
    "Test metrics already exist. Preserve and inspect the "
    "previous evaluation instead of rerunning the test."
)

checkpoint_path = Path(OUTPUT_DIR) / "best_model.pth"

assert checkpoint_path.is_file()

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

assert checkpoint["epoch"] == 10
assert checkpoint["num_classes"] == 4

manifest_hash = hashlib.sha256(
    d2_manifest_path.read_bytes()
).hexdigest()

assert checkpoint["manifest_sha256"] == manifest_hash

class_names = [
    "Bacterial Blight",
    "Blast",
    "Brown Spot",
    "Tungro"
]

assert all(
    d2_class_map[name] == index
    for index, name in enumerate(class_names)
)

class_ids = np.arange(4)

test_rows = d2_manifest.loc[
    d2_manifest["split"] == "test"
].reset_index(drop=True)

assert len(test_rows) == 713
assert set(test_rows["class_id"].astype(int)) == set(range(4))
assert test_rows["rel_path"].is_unique

test_dataset = Paper6RiceDataset(
    test_rows,
    D2_ROOT,
    Paper6Transform(training=False)
)

test_loader = DataLoader(
    test_dataset,
    batch_size=30,
    shuffle=False,
    num_workers=0,
    drop_last=False
)

assert len(test_loader) == 24

evaluation_model = Paper6MFASqueezeNet(
    num_classes=4
)

evaluation_model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

evaluation_model = evaluation_model.to(device)
evaluation_model.eval()

true_labels = []
predicted_labels = []
probability_batches = []

with torch.inference_mode():
    for images, labels in test_loader:
        images = images.to(device)

        logits = evaluation_model(images)

        assert logits.ndim == 2
        assert logits.shape[1] == 4
        assert torch.isfinite(logits).all().item()

        probabilities = torch.softmax(
            logits,
            dim=1
        )

        predictions = probabilities.argmax(dim=1)

        true_labels.extend(
            labels.numpy().tolist()
        )

        predicted_labels.extend(
            predictions.cpu().numpy().tolist()
        )

        probability_batches.append(
            probabilities.cpu().numpy()
        )

y_true = np.asarray(true_labels, dtype=int)
y_pred = np.asarray(predicted_labels, dtype=int)

y_prob = np.concatenate(
    probability_batches,
    axis=0
)

assert y_true.shape == (713,)
assert y_pred.shape == (713,)
assert y_prob.shape == (713, 4)

assert np.isfinite(y_prob).all()
assert np.allclose(
    y_prob.sum(axis=1),
    1.0,
    atol=1e-6
)

assert np.array_equal(
    y_true,
    test_rows["class_id"].to_numpy(dtype=int)
)

assert set(y_true.tolist()) == set(range(4))
assert set(y_pred.tolist()).issubset(set(range(4)))

accuracy = accuracy_score(
    y_true,
    y_pred
)

precision, recall, f1, support = (
    precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=class_ids,
        zero_division=0
    )
)

macro_precision = float(np.mean(precision))
macro_recall = float(np.mean(recall))
macro_f1 = float(np.mean(f1))

auc_macro = roc_auc_score(
    y_true,
    y_prob,
    labels=class_ids,
    multi_class="ovr",
    average="macro"
)

auc_weighted = roc_auc_score(
    y_true,
    y_prob,
    labels=class_ids,
    multi_class="ovr",
    average="weighted"
)

class_auc = []

for class_id in class_ids:
    binary_truth = (
        y_true == class_id
    ).astype(int)

    class_auc.append(
        roc_auc_score(
            binary_truth,
            y_prob[:, class_id]
        )
    )

confusion = confusion_matrix(
    y_true,
    y_pred,
    labels=class_ids
)

correct_count = int(np.trace(confusion))

assert confusion.shape == (4, 4)
assert confusion.sum() == 713
assert int(support.sum()) == 713
assert correct_count == int(
    np.sum(y_true == y_pred)
)

classwise = pd.DataFrame({
    "class_id": class_ids,
    "class_name": class_names,
    "precision": precision,
    "recall": recall,
    "f1_score": f1,
    "support": support,
    "correct": np.diag(confusion),
    "auc_ovr": class_auc
})

classwise.to_csv(
    result_dir / "classwise_metrics.csv",
    index=False
)

metrics = {
    "experiment": "Paper6_MFA_SqueezeNet_D2_Revised",
    "checkpoint_epoch": int(checkpoint["epoch"]),
    "test_samples": 713,
    "correct_predictions": correct_count,
    "num_classes": 4,
    "accuracy": float(accuracy),
    "precision_macro": macro_precision,
    "recall_macro": macro_recall,
    "f1_macro": macro_f1,
    "auc_macro_ovr": float(auc_macro),
    "auc_weighted_ovr": float(auc_weighted),
    "manifest_sha256": manifest_hash,
    "class_mapping": d2_class_map,
    "evaluation_method": "Single-label multiclass, argmax predictions, OvR ROC-AUC",
    "split_integrity_note": (
        "Recorded duplicate-aware gate passed. "
        "The latest audit did not independently verify "
        "near-duplicate groups across splits."
    )
}

metrics_path.write_text(
    json.dumps(
        metrics,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

confusion_df = pd.DataFrame(
    confusion,
    index=class_names,
    columns=class_names
)

confusion_df.index.name = "Actual class"
confusion_df.columns.name = "Predicted class"

confusion_df.to_csv(
    result_dir / "confusion_matrix.csv"
)

predictions_df = pd.DataFrame({
    "rel_path": test_rows["rel_path"].astype(str),
    "true_class_id": y_true,
    "true_class_name": [
        class_names[index] for index in y_true
    ],
    "predicted_class_id": y_pred,
    "predicted_class_name": [
        class_names[index] for index in y_pred
    ],
    "correct": y_true == y_pred
})

for class_id in class_ids:
    predictions_df[
        f"probability_class_{class_id}"
    ] = y_prob[:, class_id]

predictions_df.to_csv(
    result_dir / "test_predictions.csv",
    index=False
)

fig, ax = plt.subplots(figsize=(9, 7))

ConfusionMatrixDisplay(
    confusion_matrix=confusion,
    display_labels=class_names
).plot(
    ax=ax,
    cmap="Blues",
    values_format="d",
    colorbar=False,
    xticks_rotation=30
)

ax.set_title(
    "Paper 6 MFA-SqueezeNet — D2 Test Confusion Matrix"
)

fig.tight_layout()

fig.savefig(
    result_dir / "confusion_matrix.png",
    dpi=180
)

plt.close(fig)

fig, ax = plt.subplots(figsize=(9, 7))

for class_id, class_name in enumerate(class_names):
    binary_truth = (
        y_true == class_id
    ).astype(int)

    fpr, tpr, _ = roc_curve(
        binary_truth,
        y_prob[:, class_id]
    )

    ax.plot(
        fpr,
        tpr,
        label=f"{class_name} (AUC={class_auc[class_id]:.4f})"
    )

ax.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    color="gray"
)

ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title(
    "Paper 6 MFA-SqueezeNet — D2 One-vs-Rest ROC"
)
ax.legend()
ax.grid(alpha=0.2)

fig.tight_layout()

fig.savefig(
    result_dir / "roc_curves.png",
    dpi=180
)

plt.close(fig)

history = pd.read_csv(
    Path(OUTPUT_DIR) / "training_history.csv"
)

assert len(history) == 10

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 4.5)
)

axes[0].plot(
    history["epoch"],
    history["train_accuracy"],
    label="Training"
)

axes[0].plot(
    history["epoch"],
    history["val_accuracy"],
    label="Validation"
)

axes[0].set_title("Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(alpha=0.2)

axes[1].plot(
    history["epoch"],
    history["train_loss"],
    label="Training"
)

axes[1].plot(
    history["epoch"],
    history["val_loss"],
    label="Validation"
)

axes[1].set_title("Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Cross-Entropy Loss")
axes[1].legend()
axes[1].grid(alpha=0.2)

fig.tight_layout()

fig.savefig(
    result_dir / "training_curves.png",
    dpi=180
)

plt.close(fig)

archive_path = result_dir / "evaluation_results.zip"

files_to_archive = [
    "test_metrics.json",
    "classwise_metrics.csv",
    "confusion_matrix.csv",
    "confusion_matrix.png",
    "roc_curves.png",
    "test_predictions.csv",
    "training_curves.png"
]

with zipfile.ZipFile(
    archive_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as archive:
    for filename in files_to_archive:
        archive.write(
            result_dir / filename,
            arcname=filename
        )

with zipfile.ZipFile(archive_path) as archive:
    assert archive.testzip() is None
    assert set(files_to_archive) == set(archive.namelist())

print("Checkpoint epoch:", checkpoint["epoch"])
print("Test images:", len(y_true))
print("Correct predictions:", correct_count)

print("\nFINAL TEST METRICS")
print("Accuracy:", round(accuracy, 6))
print("Macro precision:", round(macro_precision, 6))
print("Macro recall:", round(macro_recall, 6))
print("Macro F1:", round(macro_f1, 6))
print("Macro AUC (OvR):", round(auc_macro, 6))
print("Weighted AUC (OvR):", round(auc_weighted, 6))

print("\nCLASS-WISE METRICS")
print(classwise.to_string(index=False))

print("\nCONFUSION MATRIX")
print(confusion_df.to_string())

print("\nResults directory:", result_dir)
print("Evaluation archive:", archive_path)

print("\nPAPER 6 D2 TEST EVALUATION: PASSED")


Checkpoint epoch: 10
Test images: 713
Correct predictions: 706

FINAL TEST METRICS
Accuracy: 0.990182
Macro precision: 0.990126
Macro recall: 0.989423
Macro F1: 0.989749
Macro AUC (OvR): 0.999901
Weighted AUC (OvR): 0.999907

CLASS-WISE METRICS
 class_id       class_name  precision   recall  f1_score  support  correct  auc_ovr
        0 Bacterial Blight   0.974490 0.989637  0.982005      193      191 0.999801
        1            Blast   0.986014 0.979167  0.982578      144      141 0.999805
        2       Brown Spot   1.000000 0.988889  0.994413      180      178 1.000000
        3           Tungro   1.000000 1.000000  1.000000      196      196 1.000000

CONFUSION MATRIX
Predicted class   Bacterial Blight  Blast  Brown Spot  Tungro
Actual class                                                 
Bacterial Blight               191      2           0       0
Blast                            3    141           0       0
Brown Spot                       2      0         178       0
Tungro 

In [12]:

import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
    confusion_matrix
)

root = Path(
    "/kaggle/working/Paper6_MFA_SqueezeNet_D2_Revised"
)

evaluation_dir = root / "evaluation"

metrics = json.loads(
    (evaluation_dir / "test_metrics.json").read_text(
        encoding="utf-8"
    )
)

predictions = pd.read_csv(
    evaluation_dir / "test_predictions.csv"
)

classwise = pd.read_csv(
    evaluation_dir / "classwise_metrics.csv"
)

saved_confusion = pd.read_csv(
    evaluation_dir / "confusion_matrix.csv",
    index_col=0
).to_numpy(dtype=int)

y_true = predictions["true_class_id"].to_numpy(dtype=int)
y_pred = predictions["predicted_class_id"].to_numpy(dtype=int)

probability_columns = [
    f"probability_class_{i}" for i in range(4)
]

y_prob = predictions[
    probability_columns
].to_numpy(dtype=float)

assert len(predictions) == 713
assert y_prob.shape == (713, 4)
assert np.isfinite(y_prob).all()
assert np.allclose(y_prob.sum(axis=1), 1.0, atol=1e-6)

calculated_confusion = confusion_matrix(
    y_true,
    y_pred,
    labels=list(range(4))
)

assert np.array_equal(
    saved_confusion,
    calculated_confusion
)

assert int(np.trace(calculated_confusion)) == 706

precision, recall, f1, support = (
    precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=list(range(4)),
        zero_division=0
    )
)

recalculated = {
    "accuracy": accuracy_score(y_true, y_pred),
    "precision_macro": float(np.mean(precision)),
    "recall_macro": float(np.mean(recall)),
    "f1_macro": float(np.mean(f1)),
    "auc_macro_ovr": roc_auc_score(
        y_true, y_prob,
        labels=list(range(4)),
        multi_class="ovr",
        average="macro"
    ),
    "auc_weighted_ovr": roc_auc_score(
        y_true, y_prob,
        labels=list(range(4)),
        multi_class="ovr",
        average="weighted"
    )
}

for name, value in recalculated.items():
    assert np.isclose(
        value,
        metrics[name],
        atol=1e-10,
        rtol=0
    ), f"Metric mismatch: {name}"

assert np.array_equal(
    support,
    classwise["support"].to_numpy()
)

for archive_path in [
    root / "training_backup.zip",
    evaluation_dir / "evaluation_results.zip"
]:
    assert archive_path.is_file()

    with zipfile.ZipFile(archive_path) as archive:
        assert archive.testzip() is None

print("Test images:", len(predictions))
print("Correct predictions:", np.trace(calculated_confusion))

for name, value in recalculated.items():
    print(name + ":", round(value, 6))

print("\nClass-wise support: VERIFIED")
print("Saved confusion matrix: VERIFIED")
print("Saved metric consistency: VERIFIED")
print("Both archives: PASSED")

print("\nPAPER 6 D2 FINAL EVIDENCE AUDIT: PASSED")


Test images: 713
Correct predictions: 706
accuracy: 0.990182
precision_macro: 0.990126
recall_macro: 0.989423
f1_macro: 0.989749
auc_macro_ovr: 0.999901
auc_weighted_ovr: 0.999907

Class-wise support: VERIFIED
Saved confusion matrix: VERIFIED
Saved metric consistency: VERIFIED
Both archives: PASSED

PAPER 6 D2 FINAL EVIDENCE AUDIT: PASSED
